# 03 - 推理与测试

## 任务
- **所有模型**：Baseline / SFT / DPO / GRPO 统一推理对比
- **方案1**：对比 Standard / Zero-shot COT / Few-shot COT 的效果
- **生成提交文件**：输出 CSV 提交结果

## 运行环境
- CPU: 可跑通流程，速度慢
- GPU: 批量推理，速度快（推荐）

In [ ]:
import os, sys

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("=== 调试信息 ===")
print(f"当前工作目录 (cwd): {os.getcwd()}")
print(f"当前文件: {__file__}" if '__file__' in locals() else "当前文件: 无法获取 (Jupyter 环境)")

# 最直接的方式：判断当前目录在哪里
cwd = os.getcwd()
workspace_root = None

if os.path.exists('/mnt/workspace/workspace/utils'):
    workspace_root = '/mnt/workspace/workspace'
elif os.path.exists('/mnt/workspace/utils'):
    workspace_root = '/mnt/workspace'
elif os.path.exists('../utils'):
    workspace_root = os.path.dirname(cwd)
elif os.path.exists('./utils'):
    workspace_root = cwd
elif 'workspace/notebooks' in cwd:
    workspace_root = os.path.dirname(cwd)
elif 'workspace' in cwd:
    workspace_root = cwd
else:
    workspace_root = cwd

sys.path.insert(0, workspace_root)
os.chdir(workspace_root)

print(f"\n设置后的工作目录: {os.getcwd()}")
print(f"sys.path 前两项: {sys.path[:2]}")
print(f"utils 目录是否存在: {os.path.exists(os.path.join(workspace_root, 'utils'))}")
print(f"utils 目录内容: {os.listdir(workspace_root) if os.path.exists(workspace_root) else '根目录不存在'}")

In [ ]:
import subprocess, sys
for p in ['transformers', 'peft', 'tqdm']:
    try: __import__(p)
    except: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('依赖就绪')

In [ ]:
from utils.pipeline_config import get_paths, get_device

paths = get_paths()
device = get_device()
BATCH_SIZE = 16 if device != 'cpu' else 1
MAX_NEW_TOKENS = 128  # 与 GRPO 训练配置一致，覆盖小学数学 CoT

print(f"设备: {device}, 批大小: {BATCH_SIZE}")
print(f"模型: {paths['base_model']}")
print(f"数据: {paths['data_dir']}")
print(f"输出: {paths['output_dir']}")

import os
MODEL_PATHS = {
    'base': paths['base_model'],
    'scheme2_sft': os.path.join(paths['output_dir'], 'scheme2_cot', 'final'),
    'scheme3_dpo': os.path.join(paths['output_dir'], 'scheme3_dpo', 'final'),
    'scheme4_grpo': os.path.join(paths['output_dir'], 'scheme4_grpo', 'final'),
}

for name, path in MODEL_PATHS.items():
    if name == 'base':
        exists = '✓' if os.path.exists(path) else '✗'
    else:
        exists = '✓' if os.path.exists(path) else '✗'
    print(f"  {exists} {name}: {path}")

## ===== 各方案模型准确率对比 =====

对所有方案（base、SFT、DPO、GRPO）逐一推理并计算准确率

In [ ]:
from scheme1_cot.inference import load_model, inference_with_cot
from scheme1_cot.cot_prompts import create_messages_with_cot, get_cot_prompt
from utils.common import load_json, extract_number
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
import gc
import torch

print('推理模块导入成功')

In [ ]:
test_data = load_json(os.path.join(paths['data_dir'], 'test.json'))
train_data = load_json(os.path.join(paths['data_dir'], 'train.json'))

if train_data and 'answer' in train_data[0]:
    eval_data = train_data[:400]
    has_ground_truth = True
    print(f'使用 train.json 前400条做评估（有Ground Truth）')
else:
    eval_data = test_data[:400]
    has_ground_truth = False
    print(f'使用 test.json 前400条做评估（无Ground Truth）')

print(f"评估样本: {len(eval_data)}条")
print(f"测试样本: {len(test_data)}条")

In [ ]:
# 调试：检查数据结构
print("=== 数据结构调试 ===")
if eval_data:
    first_item = eval_data[0]
    print(f"第一条数据: {first_item}")
    print(f"第一条数据类型: {type(first_item)}")
    if isinstance(first_item, dict):
        print(f"字典key: {list(first_item.keys())}")
        for k, v in first_item.items():
            print(f"  {k}: {type(v)}, value: {v}")


In [ ]:
all_model_results = {}
all_accuracies = {}

# 按推理顺序（从基础到高级）
model_order = ['base', 'scheme2_sft', 'scheme3_dpo', 'scheme4_grpo']

def load_cascaded_model(model_name, base_model_path, model_paths, device):
    """
    级联加载模型：正确处理 DPO/GRPO 需要前序 merge 的情况
    """
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path, use_fast=True, trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model_kwargs = {"trust_remote_code": True}
    if device != "cpu":
        model_kwargs["device_map"] = {"": device}
        model_kwargs["torch_dtype"] = torch.bfloat16
    else:
        model_kwargs["torch_dtype"] = torch.float32
    
    model = AutoModelForCausalLM.from_pretrained(base_model_path, **model_kwargs)
    if device == "cpu":
        model = model.to(device)
    
    if model_name == 'base':
        # Base 模型无需额外 PEFT
        pass
    elif model_name == 'scheme2_sft':
        # SFT 直接加在 base 上 → merge
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
    elif model_name == 'scheme3_dpo':
        # DPO 需要先加载 SFT → merge → 再加 DPO adapter → merge
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
        
        dpo_path = model_paths['scheme3_dpo']
        if os.path.exists(dpo_path) and os.path.exists(os.path.join(dpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, dpo_path)
            print(f"已加载 DPO PEFT: {dpo_path}")
            print(f"正在 merge DPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"DPO 权重已 merge")
    elif model_name == 'scheme4_grpo':
        # GRPO 需要 SFT → merge → DPO → merge → GRPO
        sft_path = model_paths['scheme2_sft']
        if os.path.exists(sft_path) and os.path.exists(os.path.join(sft_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, sft_path)
            print(f"已加载 SFT PEFT: {sft_path}")
            print(f"正在 merge SFT 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"SFT 权重已 merge")
        
        dpo_path = model_paths['scheme3_dpo']
        if os.path.exists(dpo_path) and os.path.exists(os.path.join(dpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, dpo_path)
            print(f"已加载 DPO PEFT: {dpo_path}")
            print(f"正在 merge DPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"DPO 权重已 merge")
        
        grpo_path = model_paths['scheme4_grpo']
        if os.path.exists(grpo_path) and os.path.exists(os.path.join(grpo_path, 'adapter_config.json')):
            model = PeftModel.from_pretrained(model, grpo_path)
            print(f"已加载 GRPO PEFT: {grpo_path}")
            print(f"正在 merge GRPO 权重...")
            model = model.merge_and_unload()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"GRPO 权重已 merge")
    
    # 全局设置 max_new_tokens，确保不受调用链影响
    model.generation_config.max_new_tokens = 128
    model.generation_config.max_length = None
    print(f"[DEBUG] generation_config: max_new_tokens={model.generation_config.max_new_tokens}, max_length={model.generation_config.max_length}")
    
    model.eval()
    return model, tokenizer

def run_inference(model, tokenizer, data, prompt_type='zero_shot',
                  batch_size=16, max_new_tokens=128):
    """自包含推理函数，直接控制 max_new_tokens，不依赖 inference.py"""
    from scheme1_cot.cot_prompts import create_messages_with_cot
    from utils.common import extract_number
    
    messages_list = [
        create_messages_with_cot(item["question"], prompt_type)
        for item in data
    ]
    
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    
    all_responses = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="批量推理"):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=512, padding_side="left").to(model.device)
        
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        for j, g in enumerate(generated):
            input_len = inputs["attention_mask"][j].sum().item()
            out = g[input_len:]
            all_responses.append(tokenizer.decode(out, skip_special_tokens=True).strip())
    
    results = []
    empty_count = 0
    for item, resp in zip(data, all_responses):
        answer = extract_number(resp)
        if not answer:
            answer = "0"
            empty_count += 1
        results.append((item["id"], answer))
    
    if empty_count > 0:
        print(f"  ⚠ {empty_count}/{len(data)} 条无有效数字，已填充 0")
    
    return results, all_responses

for model_name in model_order:
    model_path = MODEL_PATHS[model_name]
    
    # 检查模型是否存在
    if model_name == 'base':
        if not os.path.exists(MODEL_PATHS['base']):
            print(f"\n[!] {model_name} 模型不存在，跳过...")
            continue
    else:
        if not os.path.exists(model_path) or not os.path.exists(os.path.join(model_path, 'adapter_config.json')):
            print(f"\n[!] {model_name} PEFT 不存在，跳过...")
            continue
    
    print('\n' + '='*60)
    print(f"正在推理: {model_name}")
    print('='*60)
    
    # 加载模型（使用级联加载）
    print(f'加载模型: {paths["base_model"]}')
    model, tokenizer = load_cascaded_model(model_name, paths['base_model'], MODEL_PATHS, device)
    
    # GRPO 使用训练时的 prompt，其他用 zero_shot
    infer_prompt = 'grpo' if model_name == 'scheme4_grpo' else 'zero_shot'
    preds, raw_responses = run_inference(model, tokenizer, eval_data,
                                          prompt_type=infer_prompt,
                                          batch_size=BATCH_SIZE,
                                          max_new_tokens=MAX_NEW_TOKENS)
    all_model_results[model_name] = preds
    
    # 计算准确率
    if has_ground_truth:
        correct = sum(1 for (qid, pred), item in zip(preds, eval_data)
                     if pred.strip() == str(item['answer']).strip())
        acc = correct / len(preds) if preds else 0
        all_accuracies[model_name] = acc
        print(f"✓ 准确率: {acc:.2%} ({correct}/{len(preds)})")
    else:
        print(f"✓ 推理完成 (无 Ground Truth 无法计算准确率)")
    
    # 显示几个示例（含原始输出）
    print(f"  示例输出（前3条）:")
    for i in range(min(3, len(preds))):
        qid, pred = preds[i]
        raw = raw_responses[i][:200] if i < len(raw_responses) else ''
        print(f"    ID {qid}: 提取={pred} | 原始={raw}")
    
    # 清理显存
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print('\n' + '='*60)
print('各方案准确率对比')
print('='*60)

if all_accuracies:
    # 按准确率排序显示
    sorted_models = sorted(all_accuracies.items(), key=lambda x: x[1], reverse=True)
    
    # 找最佳模型
    best_name, best_acc = sorted_models[0]
    
    print(f"\n排名:")
    for i, (name, acc) in enumerate(sorted_models):
        marker = '🏆' if i == 0 else '   '
        print(f"  {marker} {i+1:2d}. {name:12s}: {acc:.2%}")
    
    # 计算相对提升
    if 'base' in all_accuracies:
        base_acc = all_accuracies['base']
        print(f"\n相对基础模型提升:")
        for name in model_order:
            if name in all_accuracies and name != 'base':
                diff = all_accuracies[name] - base_acc
                sign = '+' if diff > 0 else ''
                print(f"  {name:12s}: {sign}{diff:.2%}")

    best_prompt = 'zero_shot'
    
else:
    # 没有 ground truth 时默认用最佳模型
    best_name = None
    for name in ['scheme4_grpo', 'scheme3_dpo', 'scheme2_sft', 'base']:
        if name in all_model_results:
            best_name = name
            break
    best_prompt = 'zero_shot'
    
    print(f'无法计算准确率（无 train.json）')
    print(f'默认使用模型: {best_name}')

print(f"\n选定最佳模型: {best_name}")

## ===== 方案1：COT Prompt对比测试 =====

对比三种推理方式的效果差异（可选）

In [ ]:
from scheme1_cot.cot_prompts import get_cot_prompt

# 只在有最佳模型时才做 Prompt 对比
if best_name in all_model_results:
    print(f'加载最佳模型: {best_name}')
    model, tokenizer = load_cascaded_model(best_name, paths['base_model'], MODEL_PATHS, device)
    
    if has_ground_truth:
        offline_results = {}
        for pt in ['standard', 'zero_shot', 'few_shot']:
            print(f'测试 {pt}...')
            preds, _ = run_inference(model, tokenizer, eval_data,
                                       prompt_type=pt,
                                       batch_size=BATCH_SIZE,
                                       max_new_tokens=MAX_NEW_TOKENS)
            correct = sum(1 for (qid, pred), item in zip(preds, eval_data)
                         if pred.strip() == str(item['answer']).strip())
            acc = correct / len(preds) if preds else 0
            offline_results[pt] = acc
            print(f'  {pt:15s}: {acc:.2%} ({correct}/{len(preds)})')
        
        best_prompt = max(offline_results, key=offline_results.get)
        print(f'✓ 最佳Prompt（离线验证）: {best_prompt} ({offline_results[best_prompt]:.2%})')
    else:
        best_prompt = 'zero_shot'
else:
    best_prompt = 'zero_shot'
    print('跳过 Prompt 对比')

## ===== 生成提交文件 =====


In [ ]:
# 重新加载最佳模型（确保模型对象有效）
print(f'加载模型: {best_name}')
model, tokenizer = load_cascaded_model(best_name, paths['base_model'], MODEL_PATHS, device)

print(f"使用模型: {best_name}")
print(f"使用Prompt: {best_prompt}")
print(f"测试数据: {len(test_data)}条")

final_results, _ = run_inference(model, tokenizer, test_data,
                                   prompt_type=best_prompt,
                                   batch_size=BATCH_SIZE,
                                   max_new_tokens=MAX_NEW_TOKENS)
print(f"推理完成: {len(final_results)}条")

In [ ]:
from utils.common import save_csv

SUBMIT_PATH = os.path.join(paths['output_dir'], 'submit.csv')
os.makedirs(paths['output_dir'], exist_ok=True)
save_csv(final_results, SUBMIT_PATH)

print(f"提交文件已保存: {SUBMIT_PATH}")

print('\n结果预览（前10条）:')
for i, (qid, ans) in enumerate(final_results[:10]):
    print(f"  {qid}: {ans}")

In [ ]:
print('\n' + '='*60)
print('推理与测试完成')
print('='*60)
print(f"提交文件: {SUBMIT_PATH}")
print(f"样本总数: {len(final_results)}")
print(f"使用模型: {best_name}")
print(f"使用Prompt: {best_prompt}")